In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from IPython.display import HTML

In [4]:
def show_fig(fig):
    return HTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

region_mapping={'Africa (EI)':'África','Asia Pacific (EI)':'Asia Pacífico','Europe (EI)':'Europa','Middle East (EI)':'Medio Oriente','North America (EI)':'Norteamérica','South and Central America (EI)':'Sudamérica y Centroamérica'}
regiones=['Africa','Asia','Europe','North America','Oceania','South America','World']
tr={'Africa':'África','Asia':'Asia','Europe':'Europa','North America':'Norteamérica','Oceania':'Oceanía','South America':'Sudamérica','World':'Mundo'}
def parse_a2(path='../data/A_A2_r_230822.081459.xlsx'):
    r=pd.read_excel(path,sheet_name='A2',skiprows=5)
    yc=[c for c in r.columns if isinstance(c,(int,float))]
    r=r[r['Region and fuel'].notna()].copy()
    r['Region and fuel']=r['Region and fuel'].astype(str)
    r=r[~r['Region and fuel'].str.startswith('Data source:')].copy()
    rows=[]; cur=None
    for _,row in r.iterrows():
        lab=row['Region and fuel'].strip(); vals=row[yc]
        if vals.isna().all(): cur=lab; continue
        rows.append({'region':cur,'fuel':lab,**{int(y):row[y] for y in yc}})
    d=pd.DataFrame(rows).melt(id_vars=['region','fuel'],var_name='year',value_name='consumo_quad_btu')
    d['consumo_kwh']=d['consumo_quad_btu']*2.9307107e11
    d['fuel_es']=d['fuel'].replace({'Liquid fuels':'Combustibles líquidos','Natural gas':'Gas natural','Coal':'Carbón','Nuclear':'Nuclear','Other':'Otras','Total':'Total'})
    d['region_es']=d['region'].replace({'Americas':'Américas','Europe and Eurasia':'Europa y Eurasia','Asia Pacific':'Asia Pacífico','Africa and Middle East':'África y Medio Oriente','World':'Mundo'})
    return d
def add_split(fig,x=2026):
    fig.add_vline(x=x,line_dash='dash',line_color='black')
    fig.add_annotation(x=x-0.5,y=1.04,xref='x',yref='paper',text='Histórico',showarrow=False,xanchor='right')
    fig.add_annotation(x=x+0.5,y=1.04,xref='x',yref='paper',text='Proyección',showarrow=False,xanchor='left')

In [13]:
#| label: fig-panorama-fuentes-abs
d=pd.read_csv('../data/3960AFa8.csv',skiprows=3)
cols=[c for c in d.columns if c not in ['Año','Unidades']]
fig=px.area(d,x='Año',y=cols,labels={'value':'Energía primaria [TJ]','variable':'Fuente'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [4]:
#| label: fig-panorama-fuentes-pct
d=pd.read_csv('../data/3960AFa8.csv',skiprows=3)
cols=[c for c in d.columns if c not in ['Año','Unidades']]
fig=px.area(d,x='Año',y=cols,groupnorm='percent',labels={'value':'%','variable':'Fuente'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [14]:
#| label: fig-primaria-region-abs
d=pd.read_csv('../data/primary-energy-consumption-by-region.csv')
d['Entity']=d['Entity'].replace(region_mapping)
fig=px.area(d,x='Year',y='Primary energy consumption',color='Entity',labels={'Year':'Año','Primary energy consumption':'Energía primaria [TJ]','Entity':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [15]:
#| label: fig-primaria-region-pct
d=pd.read_csv('../data/primary-energy-consumption-by-region.csv')
d['Entity']=d['Entity'].replace(region_mapping)
fig=px.area(d,x='Year',y='Primary energy consumption',color='Entity',groupnorm='percent',labels={'Year':'Año','Primary energy consumption':'%','Entity':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [5]:
#| label: fig-renovables-escenarios
d=pd.read_csv('../data/iea_renovables_mundo_capacidad_es.csv')
s=d[d['producto']!='Total'].copy(); m=s[s['escenario']=='Caso principal'].copy(); a=s[s['escenario']=='Caso acelerado'].copy(); t=d[d['escenario']=='Meta'].copy()
prods=m['producto'].drop_duplicates().tolist(); colors=px.colors.qualitative.Plotly; cmap={p:colors[i%len(colors)] for i,p in enumerate(prods)}
fig=go.Figure()
for p in prods:
    mp=m[m['producto']==p].sort_values('anio'); ap=a[a['producto']==p].sort_values('anio')
    fig.add_trace(go.Bar(x=mp['anio'],y=mp['valor'],name=p,legendgroup=p,offsetgroup='Caso principal',marker={'color':cmap[p]}))
    fig.add_trace(go.Bar(x=ap['anio'],y=ap['valor'],name=p,legendgroup=p,offsetgroup='Caso acelerado',showlegend=False,marker={'color':cmap[p],'pattern':{'shape':'/'}}))
fig.add_hline(y=11500,line_dash='dash',line_color='black',annotation_text='Meta COP28: triplicar renovables para 2030',annotation_position='top left')
fig.add_trace(go.Scatter(x=t['anio'],y=t['valor'],mode='markers+text',text=['Ambición renovable actual 2030'],textposition='top center',marker={'size':10,'symbol':'diamond','color':'red'}))
fig.update_layout(barmode='stack',hovermode='x unified',xaxis_title='Año',yaxis_title='Capacidad instalada [GW]',legend_title='Tecnología')
show_fig(fig)

In [6]:
#| label: fig-oil-percap-region
d=pd.read_csv('https://ourworldindata.org/grapher/per-capita-oil.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
d=d[d['entity'].isin(regiones)].copy(); d['entity']=d['entity'].replace(tr)
fig=px.line(d,x='year',y='oil_per_capita__kwh',color='entity',labels={'year':'Año','oil_per_capita__kwh':'Consumo de petróleo [kWh per cápita]','entity':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [7]:
#| label: fig-elec-fuente-abs
d=pd.read_csv('https://ourworldindata.org/grapher/electricity-prod-source-stacked.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
w=d[d['entity']=='World'].copy(); cm={'other_renewables_excluding_bioenergy_generation__twh_chart_electricity_prod_source_stacked':'Otras renovables (sin bioenergía)','bioenergy_generation__twh_chart_electricity_prod_source_stacked':'Bioenergía','solar_generation__twh_chart_electricity_prod_source_stacked':'Solar','wind_generation__twh_chart_electricity_prod_source_stacked':'Eólica','hydro_generation__twh_chart_electricity_prod_source_stacked':'Hidroeléctrica','nuclear_generation__twh_chart_electricity_prod_source_stacked':'Nuclear','oil_generation__twh_chart_electricity_prod_source_stacked':'Petróleo','gas_generation__twh_chart_electricity_prod_source_stacked':'Gas','coal_generation__twh_chart_electricity_prod_source_stacked':'Carbón'}
sc=[c for c in w.columns if c in cm]; w=w.rename(columns=cm); src=[cm[c] for c in sc]
fig=px.area(w,x='year',y=src,labels={'year':'Año','value':'Generación eléctrica [TWh]','variable':'Fuente'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [8]:
#| label: fig-elec-fuente-pct
d=pd.read_csv('https://ourworldindata.org/grapher/electricity-prod-source-stacked.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
w=d[d['entity']=='World'].copy(); cm={'other_renewables_excluding_bioenergy_generation__twh_chart_electricity_prod_source_stacked':'Otras renovables (sin bioenergía)','bioenergy_generation__twh_chart_electricity_prod_source_stacked':'Bioenergía','solar_generation__twh_chart_electricity_prod_source_stacked':'Solar','wind_generation__twh_chart_electricity_prod_source_stacked':'Eólica','hydro_generation__twh_chart_electricity_prod_source_stacked':'Hidroeléctrica','nuclear_generation__twh_chart_electricity_prod_source_stacked':'Nuclear','oil_generation__twh_chart_electricity_prod_source_stacked':'Petróleo','gas_generation__twh_chart_electricity_prod_source_stacked':'Gas','coal_generation__twh_chart_electricity_prod_source_stacked':'Carbón'}
sc=[c for c in w.columns if c in cm]; w=w.rename(columns=cm); src=[cm[c] for c in sc]
fig=px.area(w,x='year',y=src,groupnorm='percent',labels={'year':'Año','value':'%','variable':'Fuente'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [11]:
#| label: fig-elec-percap-region-abs
d=pd.read_csv('https://ourworldindata.org/grapher/per-capita-electricity-generation.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
d=d[d['entity'].isin(regiones)].copy(); d=d[d['year']<=2022].copy(); d['region']=d['entity'].replace(tr)
v='per_capita_electricity_generation__kwh'
if v not in d.columns:
  cs=[c for c in d.columns if c not in ['entity','code','year']]; v=next((c for c in cs if 'per_capita' in c),cs[0])
fig=px.area(d.sort_values(['region','year']),x='year',y=v,color='region',labels={'year':'Año',v:'Generación eléctrica per cápita [kWh]','region':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [12]:
#| label: fig-elec-percap-region-pct
d=pd.read_csv('https://ourworldindata.org/grapher/per-capita-electricity-generation.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
d=d[d['entity'].isin(regiones)].copy(); d=d[d['year']<=2022].copy(); d['region']=d['entity'].replace(tr)
v='per_capita_electricity_generation__kwh'
if v not in d.columns:
  cs=[c for c in d.columns if c not in ['entity','code','year']]; v=next((c for c in cs if 'per_capita' in c),cs[0])
fig=px.area(d.sort_values(['region','year']),x='year',y=v,color='region',groupnorm='percent',labels={'year':'Año',v:'%','region':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [13]:
#| label: fig-primaria-2050-fuente-abs
dl=parse_a2(); d=dl[(dl['region']=='World')&(dl['fuel']!='Total')].copy()
fig=px.area(d.sort_values(['fuel_es','year']),x='year',y='consumo_kwh',color='fuel_es',labels={'year':'Año','consumo_kwh':'Consumo de energía primaria [kWh]','fuel_es':'Fuente'})
add_split(fig); fig.update_layout(hovermode='x unified')
show_fig(fig)

In [14]:
#| label: fig-primaria-2050-fuente-pct
dl=parse_a2(); d=dl[(dl['region']=='World')&(dl['fuel']!='Total')].copy()
fig=px.area(d.sort_values(['fuel_es','year']),x='year',y='consumo_kwh',color='fuel_es',groupnorm='percent',labels={'year':'Año','consumo_kwh':'%','fuel_es':'Fuente'})
add_split(fig); fig.update_layout(hovermode='x unified')
show_fig(fig)

In [15]:
#| label: fig-primaria-2050-region-abs
dl=parse_a2(); d=dl[(dl['fuel']=='Total')&(dl['region']!='World')].copy()
fig=px.area(d.sort_values(['region_es','year']),x='year',y='consumo_kwh',color='region_es',labels={'year':'Año','consumo_kwh':'Consumo de energía primaria [kWh]','region_es':'Región'})
add_split(fig); fig.update_layout(hovermode='x unified')
show_fig(fig)

In [16]:
#| label: fig-primaria-2050-region-pct
dl=parse_a2(); d=dl[(dl['fuel']=='Total')&(dl['region']!='World')].copy()
fig=px.area(d.sort_values(['region_es','year']),x='year',y='consumo_kwh',color='region_es',groupnorm='percent',labels={'year':'Año','consumo_kwh':'%','region_es':'Región'})
add_split(fig); fig.update_layout(hovermode='x unified')
show_fig(fig)

In [ ]:
import re
import html
import pandas as pd

# HTML fuente de la tabla (recortado a las filas de datos)
html_tabla = """
<div class="comp-content">
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/meca-oil-exports-trade-movements/">Middle East (ex Saudi Arabia)</a></div>
        <div class="it_cell mw180"><a href="/data/meca-oil-exports-trade-movements/">16 600 958</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/usa-oil-exports-trade-movements/">United States</a></div>
        <div class="it_cell mw180"><a href="/data/usa-oil-exports-trade-movements/">9 876 720</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/sau-oil-exports-trade-movements/">Saudi Arabia</a></div>
        <div class="it_cell mw180"><a href="/data/sau-oil-exports-trade-movements/">7 653 927</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/rus-oil-exports-trade-movements/">Russian Federation</a></div>
        <div class="it_cell mw180"><a href="/data/rus-oil-exports-trade-movements/">7 043 295</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/apja-oil-exports-trade-movements/">Asia Pacific (ex Japan)</a></div>
        <div class="it_cell mw180"><a href="/data/apja-oil-exports-trade-movements/">5 757 085</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/can-oil-exports-trade-movements/">Canada</a></div>
        <div class="it_cell mw180"><a href="/data/can-oil-exports-trade-movements/">5 135 022</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/scam-oil-exports-trade-movements/">S. &amp; Cent. America</a></div>
        <div class="it_cell mw180"><a href="/data/scam-oil-exports-trade-movements/">4 319 038</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/weaf-oil-exports-trade-movements/">West Africa</a></div>
        <div class="it_cell mw180"><a href="/data/weaf-oil-exports-trade-movements/">3 465 226</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20">Others</div>
        <div class="it_cell mw180">2 162 694</div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/noaf-oil-exports-trade-movements/">North Africa</a></div>
        <div class="it_cell mw180"><a href="/data/noaf-oil-exports-trade-movements/">2 151 036</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/euro-oil-exports-trade-movements/">Europe</a></div>
        <div class="it_cell mw180"><a href="/data/euro-oil-exports-trade-movements/">2 123 459</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/otci-oil-exports-trade-movements/">Other CIS</a></div>
        <div class="it_cell mw180"><a href="/data/otci-oil-exports-trade-movements/">2 122 267</a></div>
    </div>
    <div class="it_row cnshort">
        <div class="it_cell p20"><a href="/data/mex-oil-exports-trade-movements/">Mexico</a></div>
        <div class="it_cell mw180"><a href="/data/mex-oil-exports-trade-movements/">1 027 315</a></div>
    </div>
</div>
"""

# Traducciones a espanol
traduccion_paises = {
    "Middle East (ex Saudi Arabia)": "Oriente Medio (sin Arabia Saudita)",
    "United States": "Estados Unidos",
    "Saudi Arabia": "Arabia Saudita",
    "Russian Federation": "Federacion Rusa",
    "Asia Pacific (ex Japan)": "Asia Pacifico (sin Japon)",
    "Canada": "Canada",
    "S. & Cent. America": "Sudamerica y Centroamerica",
    "West Africa": "Africa Occidental",
    "Others": "Otros",
    "North Africa": "Africa del Norte",
    "Europe": "Europa",
    "Other CIS": "Otros paises de la CEI",
    "Mexico": "Mexico",
}

patron_filas = re.compile(
    r'<div class="it_row cnshort">\s*'
    r'<div class="it_cell p20">(?:<a [^>]*>)?([^<]+)(?:</a>)?</div>.*?'
    r'<div class="it_cell mw180"[^>]*>(?:<a [^>]*>)?([\d\s]+)(?:</a>)?</div>',
    re.DOTALL,
)

registros = []
for pais_raw, valor_raw in patron_filas.findall(html_tabla):
    pais_en = html.unescape(pais_raw).strip()
    pais_es = traduccion_paises.get(pais_en, pais_en)
    barriles = int(valor_raw.replace(" ", "").strip())
    registros.append({
        "pais": pais_es,
        "barriles_importados_diarios": barriles,
    })

df_importaciones = pd.DataFrame(registros)
df_importaciones

,pais,barriles_importados_diarios
0,Oriente Medio (sin Arabia Saudita),16600958
1,Estados Unidos,9876720
2,Arabia Saudita,7653927
3,Federacion Rusa,7043295
4,Asia Pacifico (sin Japon),5757085
5,Canada,5135022
6,Sudamerica y Centroamerica,4319038
7,Africa Occidental,3465226
8,Otros,2162694
9,Africa del Norte,2151036


In [21]:
import re
import html
import pandas as pd

# HTML fuente de la tabla de produccion de petroleo (barrels daily)
html_produccion = """
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/usa-oil-production/">United States</a></div>
  <div class="it_cell mw180"><a href="/data/usa-oil-production/">21 874 761</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/sau-oil-production/">Saudi Arabia</a></div>
  <div class="it_cell mw180"><a href="/data/sau-oil-production/">10 690 184</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/rus-oil-production/">Russian Federation</a></div>
  <div class="it_cell mw180"><a href="/data/rus-oil-production/">10 505 745</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/can-oil-production/">Canada</a></div>
  <div class="it_cell mw180"><a href="/data/can-oil-production/">5 926 309</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/chn-oil-production/">China</a></div>
  <div class="it_cell mw180"><a href="/data/chn-oil-production/">5 053 027</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/irn-oil-production/">Iran</a></div>
  <div class="it_cell mw180"><a href="/data/irn-oil-production/">4 619 434</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/irq-oil-production/">Iraq</a></div>
  <div class="it_cell mw180"><a href="/data/irq-oil-production/">4 517 896</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/are-oil-production/">UAE</a></div>
  <div class="it_cell mw180"><a href="/data/are-oil-production/">4 497 340</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/bra-oil-production/">Brazil</a></div>
  <div class="it_cell mw180"><a href="/data/bra-oil-production/">4 216 171</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/kwt-oil-production/">Kuwait</a></div>
  <div class="it_cell mw180"><a href="/data/kwt-oil-production/">2 779 784</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/nor-oil-production/">Norway</a></div>
  <div class="it_cell mw180"><a href="/data/nor-oil-production/">2 002 062</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/mex-oil-production/">Mexico</a></div>
  <div class="it_cell mw180"><a href="/data/mex-oil-production/">1 991 753</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/kaz-oil-production/">Kazakhstan</a></div>
  <div class="it_cell mw180"><a href="/data/kaz-oil-production/">1 900 433</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/qat-oil-production/">Qatar</a></div>
  <div class="it_cell mw180"><a href="/data/qat-oil-production/">1 862 995</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/nga-oil-production/">Nigeria</a></div>
  <div class="it_cell mw180"><a href="/data/nga-oil-production/">1 559 250</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/dza-oil-production/">Algeria</a></div>
  <div class="it_cell mw180"><a href="/data/dza-oil-production/">1 379 617</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/lby-oil-production/">Libya</a></div>
  <div class="it_cell mw180"><a href="/data/lby-oil-production/">1 183 273</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/ago-oil-production/">Angola</a></div>
  <div class="it_cell mw180"><a href="/data/ago-oil-production/">1 164 966</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/omn-oil-production/">Oman</a></div>
  <div class="it_cell mw180"><a href="/data/omn-oil-production/">1 000 589</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/ven-oil-production/">Venezuela</a></div>
  <div class="it_cell mw180"><a href="/data/ven-oil-production/">893 470</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/arg-oil-production/">Argentina</a></div>
  <div class="it_cell mw180"><a href="/data/arg-oil-production/">879 657</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/ind-oil-production/">India</a></div>
  <div class="it_cell mw180"><a href="/data/ind-oil-production/">834 682</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/idn-oil-production/">Indonesia</a></div>
  <div class="it_cell mw180"><a href="/data/idn-oil-production/">824 981</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/col-oil-production/">Colombia</a></div>
  <div class="it_cell mw180"><a href="/data/col-oil-production/">795 693</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/gbr-oil-production/">United Kingdom</a></div>
  <div class="it_cell mw180"><a href="/data/gbr-oil-production/">693 744</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/egy-oil-production/">Egypt</a></div>
  <div class="it_cell mw180"><a href="/data/egy-oil-production/">639 906</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/guy-oil-production/">Guyana</a></div>
  <div class="it_cell mw180"><a href="/data/guy-oil-production/">617 350</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/aze-oil-production/">Azerbaijan</a></div>
  <div class="it_cell mw180"><a href="/data/aze-oil-production/">592 120</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/mys-oil-production/">Malaysia</a></div>
  <div class="it_cell mw180"><a href="/data/mys-oil-production/">555 268</a></div>
</div>
<div class="it_row cnshort">
  <div class="it_cell p20"><a href="/data/ecu-oil-production/">Ecuador</a></div>
  <div class="it_cell mw180"><a href="/data/ecu-oil-production/">480 384</a></div>
</div>
"""

# Traducciones a espanol (si falta algun pais, se conserva el nombre original)
traduccion_paises = {
    "United States": "Estados Unidos",
    "Saudi Arabia": "Arabia Saudita",
    "Russian Federation": "Federacion Rusa",
    "Canada": "Canada",
    "China": "China",
    "Iran": "Iran",
    "Iraq": "Irak",
    "UAE": "Emiratos Arabes Unidos",
    "Brazil": "Brasil",
    "Kuwait": "Kuwait",
    "Norway": "Noruega",
    "Mexico": "Mexico",
    "Kazakhstan": "Kazajistan",
    "Qatar": "Qatar",
    "Nigeria": "Nigeria",
    "Algeria": "Argelia",
    "Libya": "Libia",
    "Angola": "Angola",
    "Oman": "Oman",
    "Venezuela": "Venezuela",
    "Argentina": "Argentina",
    "India": "India",
    "Indonesia": "Indonesia",
    "Colombia": "Colombia",
    "United Kingdom": "Reino Unido",
    "Egypt": "Egipto",
    "Guyana": "Guyana",
    "Azerbaijan": "Azerbaiyan",
    "Malaysia": "Malasia",
    "Ecuador": "Ecuador",
    "Thailand": "Tailandia",
    "Australia": "Australia",
    "Turkmenistan": "Turkmenistan",
    "Congo": "Congo",
    "Gabon": "Gabon",
    "Ghana": "Ghana",
    "Vietnam": "Vietnam",
    "Bahrain": "Barein",
    "Germany": "Alemania",
    "Chad": "Chad",
    "Peru": "Peru",
    "Turkey": "Turquia",
    "Brunei Darussalam": "Brunei",
    "Equatorial Guinea": "Guinea Ecuatorial",
    "Italy": "Italia",
    "Pakistan": "Pakistan",
    "France": "Francia",
    "South Sudan": "Sudan del Sur",
    "South Africa": "Sudafrica",
    "Trinidad & Tobago": "Trinidad y Tobago",
    "Netherlands": "Paises Bajos",
    "Syrian Arab Republic": "Siria",
    "Czech Republic": "Republica Checa",
    "South Korea": "Corea del Sur",
    "Cote d'Ivoire": "Costa de Marfil",
    "Cote d’Ivoire": "Costa de Marfil",
}

patron = re.compile(
    r'<div class="it_cell p20">(?:<a [^>]*>)?([^<]+?)(?:</a>)?</div>.*?'
    r'<div class="it_cell mw180"[^>]*>(?:<a [^>]*>)?([\d\s]+)(?:</a>)?</div>',
    re.DOTALL,
)

registros = []
for pais_raw, valor_raw in patron.findall(html_produccion):
    pais_en = html.unescape(pais_raw).strip()
    pais_en = re.sub(r"\s+", " ", pais_en)
    pais_es = traduccion_paises.get(pais_en, pais_en)
    barriles = int(valor_raw.replace(" ", "").strip())
    registros.append({
        "pais": pais_es,
        "barriles_producidos_diarios": barriles,
    })

df_produccion_petroleo = pd.DataFrame(registros)
df_produccion_petroleo

,pais,barriles_producidos_diarios
0,Estados Unidos,21874761
1,Arabia Saudita,10690184
2,Federacion Rusa,10505745
3,Canada,5926309
4,China,5053027
5,Iran,4619434
6,Irak,4517896
7,Emiratos Arabes Unidos,4497340
8,Brasil,4216171
9,Kuwait,2779784


In [22]:
import re
import html
import pandas as pd

# HTML fuente de la tabla de imports (Country/Region - Last)
html_imports_tabla = """
<tr>
  <td><a href="/en/indicator/australia/crude-oil-imports">Australia (Barrel/Day th)</a></td>
  <td><span>152.374</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/brazil/crude-oil-imports">Brazil (Barrel/Day th)</a></td>
  <td><span>282.333</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/canada/crude-oil-imports">Canada (Barrel/Day th)</a></td>
  <td><span>657.924</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/china/crude-oil-imports">China (Barrel/Day th)</a></td>
  <td><span>11,072.187</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/france/crude-oil-imports">France (Barrel/Day th)</a></td>
  <td><span>897.630</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/germany/crude-oil-imports">Germany (Barrel/Day th)</a></td>
  <td><span>1,689.240</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/greece/crude-oil-imports">Greece (Barrel/Day th)</a></td>
  <td><span>482.252</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/india/crude-oil-imports">India (Barrel/Day th)</a></td>
  <td><span>4,795.440</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/indonesia/crude-oil-imports">Indonesia (Barrel/Day th)</a></td>
  <td><span>321.917</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/italy/crude-oil-imports">Italy (Barrel/Day th)</a></td>
  <td><span>1,128.718</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/japan/crude-oil-imports">Japan (Barrel/Day th)</a></td>
  <td><span>2,321.610</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/malaysia/crude-oil-imports">Malaysia (Barrel/Day th)</a></td>
  <td><span>442.500</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/russia/crude-oil-imports">Russia (Barrel/Day th)</a></td>
  <td><span>0.000</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/singapore/crude-oil-imports">Singapore (Barrel/Day th)</a></td>
  <td><span>836.500</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/united-kingdom/crude-oil-imports">United Kingdom (Barrel/Day th)</a></td>
  <td><span>843.019</span></td>
</tr>
<tr>
  <td><a href="/en/indicator/united-states/crude-oil-imports">United States (Barrel/Day th)</a></td>
  <td><span>6,588.000</span></td>
</tr>
"""

traduccion_paises_imports = {
    "Australia": "Australia",
    "Brazil": "Brasil",
    "Canada": "Canada",
    "China": "China",
    "France": "Francia",
    "Germany": "Alemania",
    "Greece": "Grecia",
    "India": "India",
    "Indonesia": "Indonesia",
    "Italy": "Italia",
    "Japan": "Japon",
    "Malaysia": "Malasia",
    "Russia": "Rusia",
    "Singapore": "Singapur",
    "United Kingdom": "Reino Unido",
    "United States": "Estados Unidos",
}

patron_imports = re.compile(
    r'<a [^>]*>([^<]+?) \(Barrel/Day th\)</a>.*?'
    r'<span>([\d\.,]+)</span>',
    re.DOTALL,
)

registros_imports = []
for pais_raw, valor_raw in patron_imports.findall(html_imports_tabla):
    pais_en = html.unescape(pais_raw).strip()
    pais_es = traduccion_paises_imports.get(pais_en, pais_en)
    valor = float(valor_raw.replace(',', ''))
    registros_imports.append({
        "pais": pais_es,
        "barriles_importados_diarios": valor,
    })

df_importaciones_tabla_last = pd.DataFrame(registros_imports)
df_importaciones_tabla_last

,pais,barriles_importados_diarios
0,Australia,152.374
1,Brasil,282.333
2,Canada,657.924
3,China,11072.187
4,Francia,897.630
5,Alemania,1689.240
6,Grecia,482.252
7,India,4795.440
8,Indonesia,321.917
9,Italia,1128.718


In [23]:
#| label: tbl-top10-importadores-exportadores-productores
#| tbl-cap: "Top 10 paises importadores, exportadores y productores de petroleo"

# 1) Top 10 importadores (fuente Trading Economics: Barrel/Day th)
imp = (
    df_importaciones_tabla_last.copy()
    .sort_values("barriles_importados_diarios", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# 2) Top 10 exportadores (df_importaciones contiene exportaciones del bloque anterior)
exp = (
    df_importaciones.copy()
    .rename(columns={"barriles_importados_diarios": "barriles_exportados_diarios"})
    .sort_values("barriles_exportados_diarios", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# 3) Top 10 productores
pro = (
    df_produccion_petroleo.copy()
    .sort_values("barriles_producidos_diarios", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

ranking = pd.Series(range(1, 11), name="ranking")

def fmt_num(x, dec=0):
    return f"{x:,.{dec}f}".replace(",", "_").replace(".", ",").replace("_", ".")

# Tabla final combinada lista para QMD
tabla_top10_petroleo = pd.DataFrame({
    "ranking": ranking,
    "importador": imp["pais"],
    "importaciones (miles barriles/dia)": imp["barriles_importados_diarios"].map(lambda x: fmt_num(x, 3)),
    "exportador": exp["pais"],
    "exportaciones (barriles/dia)": exp["barriles_exportados_diarios"].map(lambda x: fmt_num(x, 0)),
    "productor": pro["pais"],
    "produccion (barriles/dia)": pro["barriles_producidos_diarios"].map(lambda x: fmt_num(x, 0)),
})

tabla_top10_petroleo

,ranking,importador,importaciones (miles barriles/dia),exportador,exportaciones (barriles/dia),productor,produccion (barriles/dia)
0,1,China,"11.072,187",Oriente Medio (sin Arabia Saudita),16.600.958,Estados Unidos,21.874.761
1,2,Estados Unidos,"6.588,000",Estados Unidos,9.876.720,Arabia Saudita,10.690.184
2,3,India,"4.795,440",Arabia Saudita,7.653.927,Federacion Rusa,10.505.745
3,4,Japon,"2.321,610",Federacion Rusa,7.043.295,Canada,5.926.309
4,5,Alemania,"1.689,240",Asia Pacifico (sin Japon),5.757.085,China,5.053.027
5,6,Italia,"1.128,718",Canada,5.135.022,Iran,4.619.434
6,7,Francia,"897,630",Sudamerica y Centroamerica,4.319.038,Irak,4.517.896
7,8,Reino Unido,"843,019",Africa Occidental,3.465.226,Emiratos Arabes Unidos,4.497.340
8,9,Singapur,"836,500",Otros,2.162.694,Brasil,4.216.171
9,10,Canada,"657,924",Africa del Norte,2.151.036,Kuwait,2.779.784


In [32]:
#| label: fig-co2-fuente-area

# Grafico de area apilada de emisiones de CO2 por fuente (Mundo)
df = pd.read_csv(
    "https://ourworldindata.org/grapher/co2-by-source.csv?v=1&csvType=full&useColumnShortNames=true",
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"},
)

co2 = df.copy()

col_year = next((c for c in co2.columns if c.lower() == "year"), None)
col_entity = next((c for c in co2.columns if c.lower() == "entity"), None)

if col_year is None:
    raise ValueError(f"No se encontro columna de ano. Columnas disponibles: {list(co2.columns)[:15]}")

if col_entity is not None:
    co2 = co2[co2[col_entity] == "World"].copy()

# Mapeo para el esquema real del dataset
mapa_fuentes = {
    "Carbon": "emissions_from_coal",
    "Petroleo": "emissions_from_oil",
    "Gas": "emissions_from_gas",
    "Cemento": "emissions_from_cement",
    "Quema en antorcha": "emissions_from_flaring",
    "Otras industrias": "emissions_from_other_industry",
}

cols_fuentes = [c for c in mapa_fuentes.values() if c in co2.columns]

if not cols_fuentes:
    raise ValueError(f"No se encontraron columnas de CO2 por fuente. Columnas disponibles: {list(co2.columns)}")

co2_largo = co2.melt(
    id_vars=[col_year],
    value_vars=cols_fuentes,
    var_name="col_original",
    value_name="co2_mt",
).dropna(subset=["co2_mt"])

mapa_inverso = {v: k for k, v in mapa_fuentes.items()}
co2_largo["fuente"] = co2_largo["col_original"].map(mapa_inverso)

fig = px.area(
    co2_largo,
    x=col_year,
    y="co2_mt",
    color="fuente",
    labels={
        col_year: "Año",
        "co2_mt": "Emisiones de CO2 (millones de toneladas)",
        "fuente": "Fuente",
    },
)
fig.update_layout(hovermode="x unified")
show_fig(fig)

In [33]:
#| label: fig-co2-fuente-pct

# Grafico de area porcentual por fuente de CO2 (Mundo)
fig = px.area(
    co2_largo.sort_values(col_year),
    x=col_year,
    y="co2_mt",
    color="fuente",
    groupnorm="percent",
    labels={
        col_year: "Ano",
        "co2_mt": "%",
        "fuente": "Fuente",
    },
)
fig.update_layout(hovermode="x unified")
show_fig(fig)

In [37]:
#| label: fig-ghg-gas-abs

# Grafico de area rellena: emisiones absolutas por gas (Mundo)
df = pd.read_csv(
    "https://ourworldindata.org/grapher/ghg-emissions-by-gas.csv?v=1&csvType=full&useColumnShortNames=true",
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"},
)

ghg = df.copy()

col_year = "year"
col_entity = "entity"
if col_entity in ghg.columns:
    ghg = ghg[ghg[col_entity] == "World"].copy()

# Columnas reales del dataset
mapa_gases = {
    "Dioxido de carbono (CO2)": "annual_emissions_co2_total",
    "Metano (CH4)": "annual_emissions_ch4_total_co2eq",
    "Oxido nitroso (N2O)": "annual_emissions_n2o_total_co2eq",
}

cols_gases = [c for c in mapa_gases.values() if c in ghg.columns]
if len(cols_gases) < 3:
    raise ValueError(f"Faltan columnas de gases. Disponibles: {list(ghg.columns)}")

ghg_largo = ghg.melt(
    id_vars=[col_year],
    value_vars=cols_gases,
    var_name="col_original",
    value_name="emisiones_mtco2e",
).dropna(subset=["emisiones_mtco2e"])

mapa_inverso = {v: k for k, v in mapa_gases.items()}
ghg_largo["gas"] = ghg_largo["col_original"].map(mapa_inverso)

fig = px.area(
    ghg_largo.sort_values(col_year),
    x=col_year,
    y="emisiones_mtco2e",
    color="gas",
    labels={
        col_year: "Ano",
        "emisiones_mtco2e": "Emisiones (MtCO2e)",
        "gas": "Gas",
    },
)
fig.update_layout(hovermode="x unified")
show_fig(fig)

In [38]:
#| label: fig-ghg-gas-pct

# Grafico de area porcentual por gas (Mundo)
fig = px.area(
    ghg_largo.sort_values(col_year),
    x=col_year,
    y="emisiones_mtco2e",
    color="gas",
    groupnorm="percent",
    labels={
        col_year: "Ano",
        "emisiones_mtco2e": "%",
        "gas": "Gas",
    },
)
fig.update_layout(hovermode="x unified")
show_fig(fig)

In [40]:
#| label: fig-emisiones
f = "../data/emisiones.csv"
d = pd.read_csv(f)
d

,Fuente,Emisiones de CO2 (g CO2e/kWh),Emisiones de NOx (g/kWh)
0,Geotermia,30.0 – 43.0,0.309
1,Eólica,11.75 – 15.98,0.06 – 2.29
2,Solar FV,39,0.06 – 2.29
3,Hidroenergía,15.0 – 366.1,0.06 – 2.29
4,Biomasa,555.2 – 653.5,1.61 – 4.65


In [17]:
#| label: fig-geotermia-mapa
dg=pd.read_csv('../data/merged_geothermal.csv')
dm=dg[['Country','2024 [1]','2025 [2]','2025 [3]']].copy(); dm.columns=['País','2024 [1]','2025 [2]','2025 [3]']
dl=dm.melt(id_vars=['País'],var_name='Año',value_name='Capacidad [MW]')
fig=px.choropleth(dl,locations='País',locationmode='country names',color='Capacidad [MW]',hover_name='País',animation_frame='Año',color_continuous_scale=px.colors.sequential.Plasma)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration']=1500
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration']=500
show_fig(fig)

/tmp/ipykernel_21391/238512454.py:5: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



In [18]:
#| label: tbl-geotermia-top10
#| tbl-cap: "Top 10 países con mayor capacidad instalada geotérmica"
dg=pd.read_csv('../data/merged_geothermal.csv')
a=dg[['Country','2024 [1]']].sort_values(by='2024 [1]',ascending=False).head(10).reset_index(drop=True)
b=dg[['Country','2025 [2]']].sort_values(by='2025 [2]',ascending=False).head(10).reset_index(drop=True)
c=dg[['Country','2025 [3]']].sort_values(by='2025 [3]',ascending=False).head(10).reset_index(drop=True)
top_10=pd.concat([a,b,c],axis=1)
top_10.columns=['País','2024 [1]','País','2025 [2]','País','2025 [3]']
top_10

,País,2024 [1],País,2025 [2],País,2025 [3]
0,United States,2725.000,United States,3733.50,United States,3953.0
1,Indonesia,2638.800,Indonesia,2431.90,Indonesia,2742.0
2,Philippines,1951.800,Philippines,1937.00,Philippines,2034.0
3,Türkiye,1734.338,Türkiye,1726.11,Türkiye,1797.0
4,New Zealand,1275.400,New Zealand,1376.70,New Zealand,1259.0
5,Mexico,998.500,Mexico,941.00,Kenya,980.0
6,Kenya,939.730,Italy,834.00,Mexico,976.0
7,Iceland,787.580,Kenya,816.50,Italy,916.0
8,Italy,771.790,Iceland,779.40,Iceland,808.0
9,Japan,489.054,Japan,618.20,Japan,607.0
